# Qwen2-5-32B evals: merged analysis

This notebook loads all CSVs in the current folder, merges them if they share the same base columns, and builds Plotly histograms to explore results by alignment, SFT condition, and test set.

Notes on filenames used here:
- `qwen2-5-32b-base_gsm8k.csv` = base model, no fine-tuning, GSM8K evals
- `em-` prefix = emergently misaligned on risky financial advice
- `em-<SFT>_<testname>.csv` = SFT experiment type and test name


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

DATA_DIR = Path('.')
MODEL_PREFIX = 'qwen2-5-32b-'

def parse_meta(path: Path):
    name = path.stem
    if name.startswith(MODEL_PREFIX):
        rest = name[len(MODEL_PREFIX):]
    else:
        rest = name

    # rest examples: base_gsm8k, em-base_gsm8k, em-risky-financial-inoculated_mmlu
    parts = rest.split('_')
    test_name = parts[-1] if parts else 'unknown'
    exp = '_'.join(parts[:-1]) if len(parts) > 1 else ''

    alignment = 'base'
    sft = 'none'
    if exp.startswith('em-'):
        alignment = 'em'
        sft = exp[len('em-'):] or 'unspecified'
    elif exp.startswith('base'):
        alignment = 'base'
        sft = 'none'
    elif exp:
        sft = exp

    return {
        'model': 'qwen2-5-32b',
        'alignment': alignment,
        'sft': sft,
        'test_name': test_name,
        'filename': path.name,
    }

csvs = sorted(DATA_DIR.glob('*.csv'))
base_cols = None
mismatched = []
dfs = []
skipped = []

for p in csvs:
    if p.stat().st_size <= 1:
        skipped.append(p.name)
        continue
    df = pd.read_csv(p)
    if df.columns.size == 0:
        skipped.append(p.name)
        continue

    if base_cols is None:
        base_cols = list(df.columns)
    elif list(df.columns) != base_cols:
        mismatched.append(p.name)
        continue

    meta = parse_meta(p)
    for k, v in meta.items():
        df[k] = v
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print('Loaded files:', [p.name for p in csvs])
print('Skipped (empty):', skipped)
print('Skipped (mismatched columns):', mismatched)
print('Base columns:', base_cols)
print('Merged shape:', merged.shape)


In [ ]:
merged.head()


In [ ]:
# Basic group stats
merged.groupby(['alignment', 'sft', 'test_name']).agg(
    count=('score', 'count'),
    mean_score=('score', 'mean'),
    std_score=('score', 'std'),
).reset_index().sort_values(['alignment', 'sft', 'test_name'])


In [ ]:
# Faceted Plotly histograms
def hist(col, facet_col='test_name', facet_row='sft', color='alignment', nbins=30):
    if col not in merged.columns:
        print(f'Missing column: {col}')
        return
    fig = px.histogram(
        merged,
        x=col,
        color=color,
        facet_col=facet_col,
        facet_row=facet_row,
        nbins=nbins,
        barmode='overlay',
        opacity=0.6,
        hover_data=['model', 'alignment', 'sft', 'test_name', 'filename'],
        title=f'{col} distribution by {facet_row} and {facet_col}',
    )
    fig.update_layout(height=350 + 200 * merged[facet_row].nunique())
    fig.show()

numeric_cols = merged.select_dtypes(include='number').columns.tolist()
numeric_cols


In [ ]:
# Run this for each numeric column
for col in numeric_cols:
    hist(col)
